### **<font color = #842B00>標題：下載財務報告書PDF到本地端<font>**

**<font color = #F39C12>操作步驟：執行 Step_1 ，到 Step_2 修改【資料起訖日】&【年份季度】→ 按上方雙箭頭圖示 → 全部重新執行 Restart<font>**

<img src="操作教學影片/使用圖片.jpg" width=50% />

**<font color = #032fa0>要安裝的套件：selenium、pandas、webdriver-manager、openpyxl<font>**

#### **Step 1：每次打開執行1次** ####

In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from datetime import datetime
import time
import random
import random
import pandas as pd
import requests
import os
import re
import json

#### **抓取公開已發布財報 → Step 2：修改公開發布的「資料起訖日」及「年份」「季度」** ####

In [ ]:
SDATE = '20250720'      # 資料起始日
EDATE = '20250831'      # 資料結束日
year = "114"            # 年份
season = "2"            # 季度


#------------------------下面不要動------------------------

SUBJECT = f"{year}年第{season}季"
folder_year = f"{year}年"
folder_season = f"Q{season}"

# API 端點
url = "https://mopsov.twse.com.tw/mops/web/ezsearch_query"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36',
    'Content-Type': 'application/x-www-form-urlencoded;charset=UTF-8'
}

# 定義函式來發送請求
def get_company_ids(typek):
    # POST請求的payload
    data = {
        'step': '00',
        'RADIO_CM': '1',  # 1 表示選擇市場類別
        'TYPEK': typek,   # 'sii' 代表上市, 'otc' 代表上櫃
        'CO_MARKET': '',  
        'CO_ID': '',      
        'lang': 'TW',      
        'AN': '',         
        'PRO_ITEM': 'E02', 
        'SUBJECT': SUBJECT,  
        'SDATE': SDATE,  
        'EDATE': EDATE,  
    }

    # 發送 POST 請求
    response = requests.post(url, headers=headers, data=data)
    response.encoding = 'utf-8'

    # 去除 BOM（UTF-8字節順序標記）並解析 JSON
    response_data = response.content.decode('utf-8-sig')  # 去除BOM
    response_data = response_data.encode('utf-8')         # 重新編碼成utf-8
    response_json = response_data.decode('utf-8')

    # 解析 JSON
    response_data = json.loads(response_json)

    # 確認是否成功回傳資料
    if response_data.get("status") == "success":
        return [item['COMPANY_ID'] for item in response_data['data']]
    else:
        print(f"回傳資料有誤 ({typek})，請檢查查詢條件或網站狀況。")
        return []

# 爬取上市（sii）和上櫃（otc）的公司代號
sii_company_ids = get_company_ids('sii')  # 上市
otc_company_ids = get_company_ids('otc')  # 上櫃

# 合併去除重複
unique_company_ids = list(set(sii_company_ids + otc_company_ids))

# 顯示結果
print("成功爬取的公司代號（上市 & 上櫃）：", unique_company_ids)

# 總共有幾個新增代碼
num_codes = len(unique_company_ids)

print(f"總共有 {num_codes} 個代碼")

In [ ]:
# 設定您檔案所在的目錄
folder_path = rf"./data/raw/pdf_reports/{folder_year}/{folder_season}"

# 獲取所有檔案名稱
file_names = os.listdir(folder_path)

# 用來儲存提取出來的 XXXX 部分
extracted_codes = []

# 正則表達式來提取檔案名稱中的股票代碼
pattern = r"_(\d+)_"

for file in file_names:
    # 使用正則表達式進行匹配
    match = re.search(pattern, file)
    if match:
        extracted_codes.append(match.group(1))

# 顯示提取出的 XXXX 部分
print("pdf檔案：", extracted_codes)

# 總共有幾個代碼
num_codes = len(extracted_codes)

print(f"總共有 {num_codes} 個代碼")

# 使用集合(set)來找出差異
pdf_set = set(extracted_codes)
all_set = set(unique_company_ids)

# 找出爬蟲代碼中有但pdf檔案中沒有的代碼
missing_from_pdf = all_set - pdf_set

# 將缺失的代碼整理成 stock_codes
stock_codes = list(missing_from_pdf)

print("爬蟲代號：",stock_codes)

# 總共有幾個新增代碼
num_codes = len(stock_codes)

print(f"總共有 {num_codes} 個代碼")

In [ ]:
chrome_options = Options()

profile = {
    "plugins.always_open_pdf_externally": True,  # 讓 PDF 直接下載，而不在瀏覽器中開啟
    "download.prompt_for_download": False,  # 禁用下載提示
    "download.extensions_to_open": "applications/pdf",  # 設置自動開啟 PDF 文件
    "download.default_directory": rf"./data/raw/pdf_reports/{folder_year}/{folder_season}",  # 指定下載路徑
    "download.directory_upgrade": True,
}

chrome_options.add_experimental_option("prefs", profile)

# 使用自動安裝驅動
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

In [ ]:
AD_year = (int(year) + 1911) * 100 + int(season)
count = 0

for stock_code in stock_codes:
    
    driver.get(f'https://doc.twse.com.tw/server-java/t57sb01?step=1&colorchg=1&co_id={stock_code}&year={year}&seamon=&mtype=A&')  # 設定路徑、股票代碼及年份

    time.sleep(random.randint(5, 10))
    
    if len(driver.find_elements(By.XPATH, f'//a[contains(., "AI4") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI4") and contains(., "{AD_year}")]'  # 合併報表(重編)
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI1") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI1") and contains(., "{AD_year}")]'  # 合併報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI5") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI5") and contains(., "{AD_year}")]'  # 個別財報(重編)	
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI2") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI2") and contains(., "{AD_year}")]'  # 個別報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI3") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI3") and contains(., "{AD_year}")]'  # 個體報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "A02") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "A02") and contains(., "{AD_year}")]'  # 母子公司合併報表
    else:
        下載 = f'//a[contains(., "A01") and contains(., "{AD_year}")]'  # 母公司財報
        
    if len(driver.find_elements(By.XPATH, 下載)) >= 1:
        driver.find_element(By.XPATH, 下載).click()
        driver.close()
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(random.randint(5, 10))
        driver.find_element(By.XPATH, 下載).click()
        time.sleep(random.randint(15, 20))

    count+=1
driver.quit()
print('迴圈執行了 %d 次' %count)

result = datetime.now().strftime("%Y-%m-%d %H:%M:%S %p")
print(result)
print("下載完成")

In [ ]:
import os
import pandas as pd
from datetime import datetime

# 來源資料夾
dirPath = rf"./data/raw/pdf_reports/{folder_year}/{folder_season}"

# 目標儲存路徑
save_dir = r"./data/raw/pdf_reports"
file_name = os.path.join(save_dir, "最新財報列表.xlsx")

# 讀取資料夾中的檔案名稱
listname = os.listdir(dirPath)

# 清洗資料：只提取第一個 "_" 與第二個 "_" 之間的股票代碼
cleaned_data = []
for item in listname:
    parts = item.split("_")
    if len(parts) > 2:
        cleaned_data.append(parts[1])  # 提取股票代碼

# 轉換為 DataFrame 並去除重複代碼
df = pd.DataFrame(cleaned_data, columns=["股票代碼"]).drop_duplicates()

# 儲存 Excel 檔案到指定的 OneDrive 資料夾
df.to_excel(file_name, index=False, header=False)  # 不要表頭

# 顯示時間戳記
result = datetime.now().strftime("%Y-%m-%d %H:%M:%S %p")
print(result)
print(f"檔案已成功儲存至: {file_name}")

#### **所有代號都爬取 → 修改「年份」及「季度」即可** #####

In [ ]:
import os
import re
import pandas as pd

# === 年度與季度 ===

year = "114"
season = "2"

#------------------以下勿動------------------------------

folder_year = f"{year}年"
folder_season = f"Q{season}"

# 讀取 Excel
excel_file_path = 'stockcode.xlsx'
df = pd.read_excel(excel_file_path, sheet_name='財報_202504')

# 確保所有代碼是字串格式
all_codes = df['代號'].astype(str)  
all_set = set(all_codes)

# 設定 PDF 檔案所在的目錄
folder_path = rf"./data/raw/pdf_reports/{folder_year}/{folder_season}"

# 獲取所有檔案名稱
file_names = os.listdir(folder_path)

# 用來儲存提取出的股票代碼
extracted_codes = []

# 正則表達式來提取檔案名稱中的數字（股票代碼）
pattern = r"_(\d+)_"

for file in file_names:
    match = re.search(pattern, file)
    if match:
        extracted_codes.append(match.group(1))

# 確保 pdf_set 也是字串格式
pdf_set = set(extracted_codes)

# 找出 all_set 中有但 pdf_set 沒有的代碼
missing_from_pdf = all_set - pdf_set
stock_codes = list(missing_from_pdf)

# 輸出結果
print("PDF 檔案代碼：", extracted_codes)
print("爬蟲應該抓取的代碼：", stock_codes)
print(f"總共有 {len(stock_codes)} 個代碼")

In [ ]:
chrome_options = Options()

profile = {
    "plugins.always_open_pdf_externally": True,  # 讓 PDF 直接下載，而不在瀏覽器中開啟
    "download.prompt_for_download": False,  # 禁用下載提示
    "download.extensions_to_open": "applications/pdf",  # 設置自動開啟 PDF 文件
    "download.default_directory": rf"./data/raw/pdf_reports/{folder_year}/{folder_season}",  # 指定下載路徑
    "download.directory_upgrade": True,
}

chrome_options.add_experimental_option("prefs", profile)

# 使用自動安裝驅動
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

In [ ]:
AD_year = (int(year) + 1911) * 100 + int(season)
count = 0

for stock_code in stock_codes:
    
    driver.get(f'https://doc.twse.com.tw/server-java/t57sb01?step=1&colorchg=1&co_id={stock_code}&year={year}&seamon=&mtype=A&')  # 設定路徑、股票代碼及年份

    time.sleep(random.randint(5, 10))
    
    if len(driver.find_elements(By.XPATH, f'//a[contains(., "AI4") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI4") and contains(., "{AD_year}")]'  # 合併報表(重編)
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI1") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI1") and contains(., "{AD_year}")]'  # 合併報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI5") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI5") and contains(., "{AD_year}")]'  # 個別財報(重編)	
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI2") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI2") and contains(., "{AD_year}")]'  # 個別報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI3") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI3") and contains(., "{AD_year}")]'  # 個體報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "A02") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "A02") and contains(., "{AD_year}")]'  # 母子公司合併報表
    else:
        下載 = f'//a[contains(., "A01") and contains(., "{AD_year}")]'  # 母公司財報
        
    if len(driver.find_elements(By.XPATH, 下載)) >= 1:
        driver.find_element(By.XPATH, 下載).click()
        driver.close()
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(random.randint(5, 10))
        driver.find_element(By.XPATH, 下載).click()
        time.sleep(random.randint(15, 20))

    count+=1
driver.quit()
print('迴圈執行了 %d 次' %count)

result = datetime.now().strftime("%Y-%m-%d %H:%M:%S %p")
print(result)
print("下載完成")

In [ ]:
import os
import pandas as pd
from datetime import datetime

# 來源資料夾
dirPath = rf"./data/raw/pdf_reports/{folder_year}/{folder_season}"

# 目標儲存路徑
save_dir = r"./data/raw/pdf_reports"
file_name = os.path.join(save_dir, "最新財報列表.xlsx")

# 讀取資料夾中的檔案名稱
listname = os.listdir(dirPath)

# 清洗資料：只提取第一個 "_" 與第二個 "_" 之間的股票代碼
cleaned_data = []
for item in listname:
    parts = item.split("_")
    if len(parts) > 2:
        cleaned_data.append(parts[1])  # 提取股票代碼

# 轉換為 DataFrame 並去除重複代碼
df = pd.DataFrame(cleaned_data, columns=["股票代碼"]).drop_duplicates()

# 儲存 Excel 檔案到指定的 OneDrive 資料夾
df.to_excel(file_name, index=False, header=False)  # 不要表頭

# 顯示時間戳記
result = datetime.now().strftime("%Y-%m-%d %H:%M:%S %p")
print(result)
print(f"檔案已成功儲存至: {file_name}")

#### **抓舊年度的資料→ 修改「年份」及「季度」即可**

In [ ]:
import os
import re
import pandas as pd

# === 年度與季度 ===

year = "112"
season = "2"

#------------------以下勿動------------------------------

folder_year = f"{year}年"
folder_season = f"Q{season}"

# 讀取 Excel
excel_file_path = 'stockcode.xlsx'
df = pd.read_excel(excel_file_path, sheet_name='股票池')

# 確保所有代碼是字串格式
all_codes = df['代號'].astype(str)  
all_set = set(all_codes)

# 設定 PDF 檔案所在的目錄
folder_path = rf"./data/raw/pdf_reports/{folder_year}/{folder_season}"

# 獲取所有檔案名稱
file_names = os.listdir(folder_path)

# 用來儲存提取出的股票代碼
extracted_codes = []

# 正則表達式來提取檔案名稱中的數字（股票代碼）
pattern = r"_(\d+)_"

for file in file_names:
    match = re.search(pattern, file)
    if match:
        extracted_codes.append(match.group(1))

# 確保 pdf_set 也是字串格式
pdf_set = set(extracted_codes)

# 找出 all_set 中有但 pdf_set 沒有的代碼
missing_from_pdf = all_set - pdf_set
stock_codes = list(missing_from_pdf)

# 輸出結果
print("PDF 檔案代碼：", extracted_codes)
print(f"總共有 {len(extracted_codes)} 個代碼")
print("爬蟲應該抓取的代碼：", stock_codes)
print(f"總共有 {len(stock_codes)} 個代碼")

In [ ]:
chrome_options = Options()

profile = {
    "plugins.always_open_pdf_externally": True,  # 讓 PDF 直接下載，而不在瀏覽器中開啟
    "download.prompt_for_download": False,  # 禁用下載提示
    "download.extensions_to_open": "applications/pdf",  # 設置自動開啟 PDF 文件
    "download.default_directory": rf"C:\Users\morin\OneDrive\桌面\工作共用專區\06_公開資訊觀測站\財報\{folder_year}\{folder_season}",  # 指定下載路徑
    "download.directory_upgrade": True,
}

chrome_options.add_experimental_option("prefs", profile)

# 使用自動安裝驅動
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

In [ ]:
AD_year = (int(year) + 1911) * 100 + int(season)
count = 0

for stock_code in stock_codes:
    
    driver.get(f'https://doc.twse.com.tw/server-java/t57sb01?step=1&colorchg=1&co_id={stock_code}&year={year}&seamon=&mtype=A&')  # 設定路徑、股票代碼及年份

    time.sleep(random.randint(5, 10))
    
    if len(driver.find_elements(By.XPATH, f'//a[contains(., "AI4") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI4") and contains(., "{AD_year}")]'  # 合併報表(重編)
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI1") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI1") and contains(., "{AD_year}")]'  # 合併報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI5") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI5") and contains(., "{AD_year}")]'  # 個別財報(重編)	
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI2") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI2") and contains(., "{AD_year}")]'  # 個別報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "AI3") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "AI3") and contains(., "{AD_year}")]'  # 個體報表
    elif len(driver.find_elements(By.XPATH, f'//a[contains(., "A02") and contains(., "{AD_year}")]')) >= 1:
        下載 = f'//a[contains(., "A02") and contains(., "{AD_year}")]'  # 母子公司合併報表
    else:
        下載 = f'//a[contains(., "A01") and contains(., "{AD_year}")]'  # 母公司財報
        
    if len(driver.find_elements(By.XPATH, 下載)) >= 1:
        driver.find_element(By.XPATH, 下載).click()
        driver.close()
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(random.randint(5, 10))
        driver.find_element(By.XPATH, 下載).click()
        time.sleep(random.randint(15, 20))

    count+=1
driver.quit()
print('迴圈執行了 %d 次' %count)

result = datetime.now().strftime("%Y-%m-%d %H:%M:%S %p")
print(result)
print("下載完成")